### ============================================
### NOTEBOOK : 04_pubmedbert_extraction.ipynb
### PARTIE 1 : Installation
### ============================================

In [2]:
import sys
print(sys.executable)

/opt/anaconda3/bin/python


In [3]:
import sys
!{sys.executable} -m pip install transformers torch datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 7.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 8.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 MB 8.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [transformers] [transformers]


In [4]:
import sys
import transformers
import torch

In [5]:
print(f"\n Versions :")
print(f"  - transformers : {transformers.__version__}")
print(f"  - torch : {torch.__version__}")
print(f"  - CUDA disponible : {torch.cuda.is_available()}")


 Versions :
  - transformers : 5.2.0
  - torch : 2.10.0
  - CUDA disponible : False


### ============================================
### PARTIE 2 : Test sur 1 exemple
### ============================================


In [6]:
from transformers import AutoTokenizer, AutoModel
import torch

print("\n" + "=" * 60)
print(" TEST PubMedBERT SUR 1 EXEMPLE")
print("=" * 60)



 TEST PubMedBERT SUR 1 EXEMPLE


In [7]:
print("chargement du modèle PubMedBERT...")
model_name = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
print(f" Modèle chargé : {model_name}")

chargement du modèle PubMedBERT...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Modèle chargé : microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext


### --------------------------------------------
### 2. TESTER SUR 1 NOTE
### --------------------------------------------


In [12]:
# Charger une note d'exemple 
import pandas as pd
train_data=pd.read_csv("../data/processed/train_data.csv")
exemple=train_data.iloc[0]
print(f"\n Note test :")
print(f"  - SUBJECT_ID : {exemple['SUBJECT_ID']}")
print(f"  - Longueur : {exemple['text_length']} caractères")
print(f"  - Persona : {exemple['persona_target']}")


 Note test :
  - SUBJECT_ID : 16143
  - Longueur : 184 caractères
  - Persona : Urgentiste


In [13]:
# Prendre les 512 premiers caractères (limite BERT)
text = exemple['TEXT_CLEAN'][:512]
print(f"\n Texte à analyser (512 premiers car) :")
print(text[:200] + "...\n")


 Texte à analyser (512 premiers car) :
Sinus rhythm Marked left axis deviation Poor R wave progression and Q -waves in leads V1-V2 consistent with old septal infarct Nonspecific lateral ST-T wave changes No previous tracing...



### --------------------------------------------
### 3. TOKENIZATION
### --------------------------------------------

In [14]:
print(" Tokenization...")

inputs = tokenizer(
    text,
    return_tensors="pt",  # Format PyTorch
    truncation=True,
    max_length=512,
    padding=True
)

print(f" Tokens : {inputs['input_ids'].shape}")

 Tokenization...
 Tokens : torch.Size([1, 36])


### --------------------------------------------
### 4. EXTRACTION EMBEDDINGS
### --------------------------------------------

In [15]:
print("\nExtraction embeddings...")

with torch.no_grad():  # Pas besoin de gradient (inference)
    outputs = model(**inputs)

# Récupérer embeddings
embeddings = outputs.last_hidden_state  # [1, seq_len, 768]
pooled_embedding = embeddings.mean(dim=1)  # [1, 768] (moyenne)

print(f" Embeddings extraits : {pooled_embedding.shape}")
print(f"   Exemple valeurs : {pooled_embedding[0][:10]}")



Extraction embeddings...
 Embeddings extraits : torch.Size([1, 768])
   Exemple valeurs : tensor([-0.1691,  0.1931,  0.0453, -0.1298,  0.0333,  0.1255,  0.0522,  0.1427,
         0.2985, -0.0140])


### --------------------------------------------
### 5. VÉRIFICATION
### --------------------------------------------


In [17]:
print("\n TEST RÉUSSI !")
print(f"   PubMedBERT fonctionne correctement sur le texte !")
print(f"   Embedding dimension : {pooled_embedding.shape[1]} (attendu: 768)")


 TEST RÉUSSI !
   PubMedBERT fonctionne correctement sur le texte !
   Embedding dimension : 768 (attendu: 768)
